# blipb quickstart

One comparator evaluation at the SPEC baseline (STARC-ABL-class fuselage, M0.785 / FL350),
the exact ledger check, the Hall-style decomposition, and the mission conversion.

Run `uv sync --extra dev` first, then open this notebook with the project kernel
(`uv run jupyter lab` after `uv pip install jupyterlab`, or use VS Code).

In [ ]:
from blipb import BLIComparator
from blipb.powerbalance import hall2017

comp = BLIComparator()  # solves the fuselage boundary layer once (~0.1 s)
bl = comp.bl
print(f"Re_L = {comp.flight.reynolds(comp.fuselage.length):.2e}")
print(f"theta_TE = {bl.theta_te*1e3:.0f} mm, H_TE = {bl.h_te:.2f}, D_fus = {bl.drag/1e3:.2f} kN")

In [ ]:
res = comp.run_design(f_phi=0.5, fpr=1.25)
print(f"subsystem PSC          = {res.psc:.2%}")
print(f"ledger residual        = {res.ledger_residual:.1e}  (exact identity, Appendix A)")
print(f"decomposition          = {hall2017.decompose(res).as_dict()}")
print(f"net PSC (turboelectric) = {comp.net_psc(res, phi=0.28, eta_elec=0.92):.2%}")

In [ ]:
# Sweep ingestion fraction and plot the dilution effect vs the absolute saving
import matplotlib.pyplot as plt
import numpy as np

fs = np.linspace(0.05, 0.95, 40)
runs = [comp.run_design(f_phi=f, fpr=1.25) for f in fs]
fig, ax1 = plt.subplots(figsize=(6, 3))
ax1.plot(fs, [r.psc * 100 for r in runs], color="#0072B2")
ax1.set_xlabel("$f_\\Phi$"); ax1.set_ylabel("subsystem PSC [%]", color="#0072B2")
ax2 = ax1.twinx()
ax2.plot(fs, [(r.pk_pod - r.pk_bli) / 1e3 for r in runs], color="#D55E00")
ax2.set_ylabel("absolute saving [kW]", color="#D55E00")
plt.title("dilution effect: the ratio falls while the saving grows")
plt.show()

Next steps: `studies/atlas.py` for the full maps, `validation/make_table.py` for the
per-rule validation table, `studies/run_uq_pilot.py` for Sobol + PCE + Monte-Carlo.